# Training Loss Curves

In [1]:
import pickle
import os

import jax.numpy as jnp

from src.functions import *
from src.pdes import *
from src.utils import *

from jaxkan.KAN import KAN

from flax import nnx
import optax

from sklearn.model_selection import train_test_split

## Function Fitting

We proceed with the training of the two networks mentioned in the manuscript to show the evolution of the training loss for each function, under the selected initialization techniques.

In [2]:
# Setup
func_dict = {"f1": f1, "f2": f2, "f3": f3, "f4": f4, "f5": f5}

N = 5000
seed = 42

num_epochs = 2000

init_lr = 0.001
transition_steps = 50
#warmup_steps = 200
decay_rate = 0.9

lr_schedule = optax.exponential_decay(init_value=init_lr, transition_steps=transition_steps, decay_rate=decay_rate, staircase=False)

opt_type = optax.adam(learning_rate = lr_schedule)


pow_basis = 1.75
pow_res = 0.25

# --------------------------
# Small architecture details
# --------------------------
G_small = 5
hidden_small = [8, 8]

params_small_baseline = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'default'}}

params_small_lecun = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'lecun', 'gain': None, 'distribution': 'uniform'}}

params_small_glorot = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'glorot', 'gain': None, 'distribution': 'uniform'}}

params_small_power = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

# ------------------------
# Big architecture details
# ------------------------
G_big = 20
hidden_big = [32, 32, 32]

params_big_baseline = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'default'}}

params_big_lecun = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'lecun', 'gain': None, 'distribution': 'uniform'}}

params_big_glorot = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'glorot', 'gain': None, 'distribution': 'uniform'}}

params_big_power = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}


In [3]:
# Initialize results dict
results = dict()

for func_name in func_dict.keys():
    print(f"Running Experiments for {func_name}.")
    function = func_dict[func_name]
    results[func_name] = dict()

    results[func_name]['small'] = dict()
    results[func_name]['big'] = dict()

    # Generate data
    x, y = generate_func_data(function, 2, N, seed)

    # Split data (in this case we do not care about mse loss, but we're doing it for consistency)
    X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=seed)

    # Model input/output
    n_in, n_out = X_train.shape[1], y_train.shape[1]

    # Small architecture
    layer_dims = [n_in, *hidden_small, n_out]

    print(f"\tTraining model with dimensions {layer_dims}.")

    # For confidence
    for run in [1, 2, 3, 4, 5]:

        results[func_name]['small'][run] = dict()

        print(f"\t\tRun No. {run}.")

        # Baseline
        base_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_baseline, seed = seed+run)
        base_opt = nnx.Optimizer(base_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(base_model, base_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['small'][run]['baseline'] = train_losses.copy()

        print(f"\t\t\tBaseline model: Final Loss = {loss:.2e}")

        # LeCun
        lecun_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_lecun, seed = seed+run)
        lecun_opt = nnx.Optimizer(lecun_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(lecun_model, lecun_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['small'][run]['lecun'] = train_losses.copy()

        print(f"\t\t\tLeCun model: Final Loss = {loss:.2e}")

        # Glorot
        glorot_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_glorot, seed = seed+run)
        glorot_opt = nnx.Optimizer(glorot_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(glorot_model, glorot_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['small'][run]['glorot'] = train_losses.copy()

        print(f"\t\t\tGlorot model: Final Loss = {loss:.2e}")

        # Power Law
        power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_power, seed = seed+run)
        power_opt = nnx.Optimizer(power_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(power_model, power_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['small'][run]['power'] = train_losses.copy()

        print(f"\t\t\tPower-law model: Final Loss = {loss:.2e}")

    # Big architecture
    layer_dims = [n_in, *hidden_big, n_out]

    print(f"\tTraining model with dimensions {layer_dims}.")

    # For confidence
    for run in [1, 2, 3, 4, 5]:

        results[func_name]['big'][run] = dict()

        print(f"\t\tRun No. {run}.")

        # Baseline
        base_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_baseline, seed = seed+run)
        base_opt = nnx.Optimizer(base_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(base_model, base_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['big'][run]['baseline'] = train_losses.copy()

        print(f"\t\t\tBaseline model: Final Loss = {loss:.2e}")

        # LeCun
        lecun_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_lecun, seed = seed+run)
        lecun_opt = nnx.Optimizer(lecun_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(lecun_model, lecun_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['big'][run]['lecun'] = train_losses.copy()

        print(f"\t\t\tLeCun model: Final Loss = {loss:.2e}")

        # Glorot
        glorot_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_glorot, seed = seed+run)
        glorot_opt = nnx.Optimizer(glorot_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(glorot_model, glorot_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['big'][run]['glorot'] = train_losses.copy()

        print(f"\t\t\tGlorot model: Final Loss = {loss:.2e}")

        # Power Law
        power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_power, seed = seed+run)
        power_opt = nnx.Optimizer(power_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss = func_fit_step(power_model, power_opt, X_train, y_train)
            train_losses = train_losses.at[epoch].set(loss)

        results[func_name]['big'][run]['power'] = train_losses.copy()

        print(f"\t\t\tPower-law model: Final Loss = {loss:.2e}")

Running Experiments for f1.
	Training model with dimensions [2, 8, 8, 1].
		Run No. 1.


2025-11-14 11:09:35.343207: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


			Baseline model: Final Loss = 6.86e-05
			LeCun model: Final Loss = 1.61e-03
			Glorot model: Final Loss = 1.80e-04
			Power-law model: Final Loss = 7.96e-06
		Run No. 2.
			Baseline model: Final Loss = 3.42e-05
			LeCun model: Final Loss = 8.33e-04
			Glorot model: Final Loss = 9.40e-05
			Power-law model: Final Loss = 7.45e-06
		Run No. 3.
			Baseline model: Final Loss = 3.78e-05
			LeCun model: Final Loss = 2.06e-03
			Glorot model: Final Loss = 1.49e-04
			Power-law model: Final Loss = 8.44e-06
		Run No. 4.
			Baseline model: Final Loss = 5.23e-05
			LeCun model: Final Loss = 2.37e-03
			Glorot model: Final Loss = 8.94e-05
			Power-law model: Final Loss = 8.39e-06
		Run No. 5.
			Baseline model: Final Loss = 6.94e-05
			LeCun model: Final Loss = 2.27e-03
			Glorot model: Final Loss = 1.65e-04
			Power-law model: Final Loss = 7.38e-06
	Training model with dimensions [2, 32, 32, 32, 1].
		Run No. 1.


2025-11-14 11:10:58.394986: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


			Baseline model: Final Loss = 1.19e-05
			LeCun model: Final Loss = 2.36e-04
			Glorot model: Final Loss = 6.08e-06
			Power-law model: Final Loss = 4.51e-08
		Run No. 2.
			Baseline model: Final Loss = 9.86e-06
			LeCun model: Final Loss = 2.04e-04
			Glorot model: Final Loss = 4.29e-06
			Power-law model: Final Loss = 4.59e-08
		Run No. 3.
			Baseline model: Final Loss = 1.19e-05
			LeCun model: Final Loss = 1.91e-04
			Glorot model: Final Loss = 3.79e-06
			Power-law model: Final Loss = 4.55e-08
		Run No. 4.
			Baseline model: Final Loss = 1.53e-05
			LeCun model: Final Loss = 2.17e-04
			Glorot model: Final Loss = 5.84e-06
			Power-law model: Final Loss = 5.36e-08
		Run No. 5.
			Baseline model: Final Loss = 1.50e-05
			LeCun model: Final Loss = 2.13e-04
			Glorot model: Final Loss = 6.45e-06
			Power-law model: Final Loss = 4.46e-08
Running Experiments for f2.
	Training model with dimensions [2, 8, 8, 1].
		Run No. 1.
			Baseline model: Final Loss = 7.59e-03
			LeCun model: Fina

In [4]:
# Save results for further processing
results_dir = 'ff_results/'

with open(os.path.join(results_dir, "losses_lr.pkl"), "wb") as f:
    pickle.dump(results, f)

## PDE

And likewise for the PDEs.

In [5]:
# Setup
pde_dict = {"allen-cahn": ac_res, "burgers": burgers_res, "helmholtz": helmholtz_res}

N_points = 2**6

RBA_gamma = 0.999
RBA_eta = 0.01

seed = 42

num_epochs = 5000

init_lr = 0.001
transition_steps = 100
decay_rate = 0.85

lr_schedule = optax.exponential_decay(init_value=init_lr, transition_steps=transition_steps, decay_rate=decay_rate, staircase=False)
opt_type = optax.adam(learning_rate = lr_schedule)

pow_basis = 1.75
pow_res = 0.25

# --------------------------
# Small architecture details
# --------------------------
G_small = 5
hidden_small = [8, 8]

params_small_baseline = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'default'}}

params_small_lecun = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'lecun', 'gain': None, 'distribution': 'uniform'}}

params_small_glorot = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'glorot', 'gain': None, 'distribution': 'uniform'}}

params_small_power = {'k': 3, 'G': G_small, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

# ------------------------
# Big architecture details
# ------------------------
G_big = 20
hidden_big = [32, 32, 32]

params_big_baseline = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                         'init_scheme': {'type': 'default'}}

params_big_lecun = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'lecun', 'gain': None, 'distribution': 'uniform'}}

params_big_glorot = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                            'init_scheme': {'type': 'glorot', 'gain': None, 'distribution': 'uniform'}}

params_big_power = {'k': 3, 'G': G_big, 'grid_range': (-1.0, 1.0), 'grid_e': 1.0, 'residual': nnx.silu, 'external_weights': True, 'add_bias': True,
                      'init_scheme': {'type': 'power', "const_b": 1.0, "const_r": 1.0, "pow_b1": pow_basis, "pow_b2": pow_basis, "pow_r1": pow_res, "pow_r2": pow_res}}

In [6]:
# Experiment
# Initialize results dict
results = dict()

for pde_name in pde_dict.keys():
    print(f"Running Experiments for {pde_name} equation.")
    pde_res = pde_dict[pde_name]
    results[pde_name] = dict()

    results[pde_name]['small'] = dict()
    results[pde_name]['big'] = dict()

    # Define the loss function for this PDE
    def loss_fn(model, l_E, l_B, pde_collocs, bc_collocs, bc_data):

        # ------------- PDE ---------------------------- #
        pde_residuals = pde_res(model, pde_collocs)
    
        # Get new RBA weights
        abs_pde_res = jnp.abs(pde_residuals)
        l_E_new = (RBA_gamma*l_E) + (RBA_eta*abs_pde_res/jnp.max(abs_pde_res))
    
        # Multiply by RBA weights
        w_resids_pde = l_E_new * pde_residuals
    
        # Get loss
        pde_loss = jnp.mean(w_resids_pde**2)
    
    
        # ------------- BC ----------------------------- #
        bc_residuals = model(bc_collocs) - bc_data
    
        # Get new RBA weights
        abs_bc_res = jnp.abs(bc_residuals)
        l_B_new = (RBA_gamma*l_B) + (RBA_eta*abs_bc_res/jnp.max(abs_bc_res))
    
        # Multiply by RBA weights
        w_resids_bc = l_B_new * bc_residuals
    
        # Loss
        bc_loss = jnp.mean(w_resids_bc**2)
    
        
        # ------------- Total --------------------------- #
        total_loss = pde_loss + bc_loss
    
        return total_loss, (l_E_new, l_B_new)
        
    # Define the train step
    @nnx.jit
    def train_step(model, optimizer, l_E, l_B, pde_collocs, bc_collocs, bc_data):
    
        (loss, (l_E_new, l_B_new)), grads = nnx.value_and_grad(loss_fn, has_aux = True)(model, l_E, l_B, pde_collocs, bc_collocs, bc_data)
    
        optimizer.update(grads)
    
        return loss, l_E_new, l_B_new

    # Get the reference solution
    refsol, coords = get_ref(pde_name)

    # Get collocation points
    pde_collocs, bc_collocs, bc_data = get_collocs(pde_name, N_points)

    # Model input/output
    n_in, n_out = pde_collocs.shape[1], bc_data.shape[1]

    # Small architecture
    layer_dims = [n_in, *hidden_small, n_out]

    print(f"\tTraining model with dimensions {layer_dims}.")

    # For confidence
    for run in [1, 2, 3, 4, 5]:

        results[pde_name]['small'][run] = dict()

        print(f"\t\tRun No. {run}.")

        # Baseline
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        base_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_baseline, seed = seed+run)
        base_opt = nnx.Optimizer(base_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(base_model, base_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['small'][run]['baseline'] = train_losses.copy()

        print(f"\t\t\tBaseline model: Final Loss = {loss:.2e}")

        # LeCun
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        lecun_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_lecun, seed = seed+run)
        lecun_opt = nnx.Optimizer(lecun_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(lecun_model, lecun_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['small'][run]['lecun'] = train_losses.copy()

        print(f"\t\t\tLeCun model: Final Loss = {loss:.2e}")

        # Glorot
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        glorot_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_glorot, seed = seed+run)
        glorot_opt = nnx.Optimizer(glorot_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(glorot_model, glorot_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['small'][run]['glorot'] = train_losses.copy()

        print(f"\t\t\tGlorot model: Final Loss = {loss:.2e}")

        # Power Law
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_small_power, seed = seed+run)
        power_opt = nnx.Optimizer(power_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(power_model, power_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['small'][run]['power'] = train_losses.copy()

        print(f"\t\t\tPower-law model: Final Loss = {loss:.2e}")

    # Big architecture
    layer_dims = [n_in, *hidden_big, n_out]

    print(f"\tTraining model with dimensions {layer_dims}.")

    # For confidence
    for run in [1, 2, 3, 4, 5]:

        results[pde_name]['big'][run] = dict()

        print(f"\t\tRun No. {run}.")

        # Baseline
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        base_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_baseline, seed = seed+run)
        base_opt = nnx.Optimizer(base_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(base_model, base_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['big'][run]['baseline'] = train_losses.copy()

        print(f"\t\t\tBaseline model: Final Loss = {loss:.2e}")

        # LeCun
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        lecun_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_lecun, seed = seed+run)
        lecun_opt = nnx.Optimizer(lecun_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(lecun_model, lecun_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['big'][run]['lecun'] = train_losses.copy()

        print(f"\t\t\tLeCun model: Final Loss = {loss:.2e}")

        # Glorot
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        glorot_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_glorot, seed = seed+run)
        glorot_opt = nnx.Optimizer(glorot_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(glorot_model, glorot_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['big'][run]['glorot'] = train_losses.copy()

        print(f"\t\t\tGlorot model: Final Loss = {loss:.2e}")

        # Power Law
        l_E = jnp.ones((pde_collocs.shape[0], 1))
        l_B = jnp.ones((bc_collocs.shape[0], 1))
        
        power_model = KAN(layer_dims = layer_dims, layer_type = 'spline', required_parameters = params_big_power, seed = seed+run)
        power_opt = nnx.Optimizer(power_model, opt_type)

        train_losses = jnp.zeros((num_epochs,))
        for epoch in range(num_epochs):
            loss, l_E, l_B = train_step(power_model, power_opt, l_E, l_B, pde_collocs, bc_collocs, bc_data)
            train_losses = train_losses.at[epoch].set(loss)

        results[pde_name]['big'][run]['power'] = train_losses.copy()

        print(f"\t\t\tPower-law model: Final Loss = {loss:.2e}")

Running Experiments for allen-cahn equation.
	Training model with dimensions [2, 8, 8, 1].
		Run No. 1.


2025-11-14 11:27:01.511409: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 11:27:01.511471: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


			Baseline model: Final Loss = 4.37e-02
			LeCun model: Final Loss = 1.70e-01
			Glorot model: Final Loss = 7.42e-02
			Power-law model: Final Loss = 2.87e-02
		Run No. 2.
			Baseline model: Final Loss = 3.63e-02
			LeCun model: Final Loss = 1.77e-01
			Glorot model: Final Loss = 5.24e-02
			Power-law model: Final Loss = 3.01e-02
		Run No. 3.
			Baseline model: Final Loss = 3.52e-02
			LeCun model: Final Loss = 1.39e-01
			Glorot model: Final Loss = 5.88e-02
			Power-law model: Final Loss = 3.05e-02
		Run No. 4.
			Baseline model: Final Loss = 3.97e-02
			LeCun model: Final Loss = 4.00e-01
			Glorot model: Final Loss = 4.91e-02
			Power-law model: Final Loss = 2.73e-02
		Run No. 5.
			Baseline model: Final Loss = 4.29e-02
			LeCun model: Final Loss = 7.53e-02
			Glorot model: Final Loss = 5.47e-02
			Power-law model: Final Loss = 2.65e-02
	Training model with dimensions [2, 32, 32, 32, 1].
		Run No. 1.


2025-11-14 11:31:20.731146: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 11:31:20.731203: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 11:31:20.731214: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 11:31:20.731222: W external/xla/xla/service/gpu/au

			Baseline model: Final Loss = 3.73e+00
			LeCun model: Final Loss = 6.70e+03
			Glorot model: Final Loss = 1.68e-02
			Power-law model: Final Loss = 3.18e-04
		Run No. 2.
			Baseline model: Final Loss = 3.11e+00
			LeCun model: Final Loss = 4.32e+03
			Glorot model: Final Loss = 1.55e-02
			Power-law model: Final Loss = 1.89e-04
		Run No. 3.
			Baseline model: Final Loss = 3.09e+00
			LeCun model: Final Loss = 5.74e+03
			Glorot model: Final Loss = 1.52e-02
			Power-law model: Final Loss = 7.70e-04
		Run No. 4.
			Baseline model: Final Loss = 4.77e+00
			LeCun model: Final Loss = 6.08e+03
			Glorot model: Final Loss = 1.52e-02
			Power-law model: Final Loss = 2.18e-04
		Run No. 5.
			Baseline model: Final Loss = 4.02e+00
			LeCun model: Final Loss = 6.35e+03
			Glorot model: Final Loss = 1.71e-02
			Power-law model: Final Loss = 2.99e-04
Running Experiments for burgers equation.
	Training model with dimensions [2, 8, 8, 1].
		Run No. 1.


...

			Baseline model: Final Loss = 9.35e-02
			LeCun model: Final Loss = 2.16e+00
			Glorot model: Final Loss = 1.34e-01
			Power-law model: Final Loss = 7.68e-02
		Run No. 2.
			Baseline model: Final Loss = 1.28e-01
			LeCun model: Final Loss = 1.90e+00
			Glorot model: Final Loss = 1.78e-01
			Power-law model: Final Loss = 1.27e-01
		Run No. 3.
			Baseline model: Final Loss = 7.74e-02
			LeCun model: Final Loss = 2.01e+00
			Glorot model: Final Loss = 1.82e-01
			Power-law model: Final Loss = 6.98e-02
		Run No. 4.
			Baseline model: Final Loss = 1.42e-01
			LeCun model: Final Loss = 2.16e+00
			Glorot model: Final Loss = 9.65e-02
			Power-law model: Final Loss = 5.68e-02
		Run No. 5.
			Baseline model: Final Loss = 9.75e-02
			LeCun model: Final Loss = 2.24e+00
			Glorot model: Final Loss = 2.40e-01
			Power-law model: Final Loss = 9.11e-02
	Training model with dimensions [2, 32, 32, 32, 1].
		Run No. 1.
			Baseline model: Final Loss = 2.06e+01
			LeCun model: Final Loss = 2.29e+06
			G

2025-11-14 12:14:39.316420: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.


			Baseline model: Final Loss = 3.55e+02
			LeCun model: Final Loss = 6.74e+02
			Glorot model: Final Loss = 3.41e+02
			Power-law model: Final Loss = 2.83e+02
		Run No. 2.
			Baseline model: Final Loss = 3.00e+02
			LeCun model: Final Loss = 5.32e+02
			Glorot model: Final Loss = 3.76e+02
			Power-law model: Final Loss = 3.10e+02
		Run No. 3.
			Baseline model: Final Loss = 3.00e+02
			LeCun model: Final Loss = 1.06e+03
			Glorot model: Final Loss = 2.15e+02
			Power-law model: Final Loss = 2.22e+02
		Run No. 4.
			Baseline model: Final Loss = 4.24e+02
			LeCun model: Final Loss = 4.64e+02
			Glorot model: Final Loss = 2.26e+02
			Power-law model: Final Loss = 1.71e+02
		Run No. 5.
			Baseline model: Final Loss = 2.69e+02
			LeCun model: Final Loss = 5.99e+02
			Glorot model: Final Loss = 2.56e+02
			Power-law model: Final Loss = 3.71e+02
	Training model with dimensions [2, 32, 32, 32, 1].
		Run No. 1.


2025-11-14 12:19:23.771806: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 12:19:23.771867: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 12:19:23.771881: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-11-14 12:19:23.771889: W external/xla/xla/service/gpu/au

			Baseline model: Final Loss = 2.76e+06
			LeCun model: Final Loss = 5.39e+11
			Glorot model: Final Loss = 1.33e+00
			Power-law model: Final Loss = 5.29e-01
		Run No. 2.
			Baseline model: Final Loss = 1.79e+06
			LeCun model: Final Loss = 4.57e+11
			Glorot model: Final Loss = 1.60e+00
			Power-law model: Final Loss = 8.52e-01
		Run No. 3.
			Baseline model: Final Loss = 2.37e+06
			LeCun model: Final Loss = 5.35e+11
			Glorot model: Final Loss = 1.22e+00
			Power-law model: Final Loss = 9.44e-01
		Run No. 4.
			Baseline model: Final Loss = 3.15e+06
			LeCun model: Final Loss = 5.51e+11
			Glorot model: Final Loss = 1.19e+00
			Power-law model: Final Loss = 1.02e+00
		Run No. 5.
			Baseline model: Final Loss = 3.12e+06
			LeCun model: Final Loss = 5.95e+11
			Glorot model: Final Loss = 1.50e+00
			Power-law model: Final Loss = 8.36e-01


In [7]:
# Save results for further processing
results_dir = 'pde_results/'

with open(os.path.join(results_dir, "losses_lr.pkl"), "wb") as f:
    pickle.dump(results, f)